<a href="https://colab.research.google.com/github/Andru-1987/data_science_ii_96085/blob/main/15_semana/clase_teorica/modelos_de_ensambles.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Ensambles aplicados: predicción de listings inactivos en Airbnb Buenos Aires

**Contexto.** Una plataforma quiere detectar alojamientos sin actividad en los últimos 12 meses.
Un listing es *inactivo* si `number_of_reviews_ltm == 0`.

**Objetivo.**
1. Análisis estadístico exploratorio que justifique las decisiones de modelado.
2. Comparar Bagging, AdaBoost y Gradient Boosting con CV solo sobre train.
3. Optimizar el mejor candidato con `RandomizedSearchCV` | `HalvingSearch`.
4. Elegir el umbral con predicciones out-of-fold, no con test.
5. Evaluar una única vez sobre test e interpretar.

## Parte 1. Carga y calidad de datos

Observar: `neighbourhood_group` y `license` casi vacías; `price` con n
ulos;
`reviews_per_month` y `last_review` nulos cuando no hay reseñas (relevante para el leakage).

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats
import matplotlib.pyplot as plt

In [ ]:

URL = "https://raw.githubusercontent.com/Andru-1987/csv_files_ds/refs/heads/main/buenos_aires.csv"
df = pd.read_csv(URL)

print(f"Shape de los datos: {df.shape}")
calidad = pd.DataFrame({
    "nulos": df.isna().sum(),
    "pct_nulos": (df.isna().mean() * 100).round(1),
    "unicos": df.nunique(),
})

calidad.sort_values("pct_nulos", ascending=False)

In [ ]:
df = df.drop_duplicates("id")
df = df.dropna(subset=["price"])
df = df[df["price"] > 0].copy()

df["inactivo"] = (df["number_of_reviews_ltm"] == 0).astype(int)
df["log_price"] = np.log1p(df["price"])

print(f"Filas tras limpieza: {len(df)}")
print(f"Proporción de inactivos: {df['inactivo'].mean():.3f}")

**Pregunta.** ¿Qué tan desbalanceado está el target? Esa proporción decide si accuracy sirve
como métrica y qué baseline usar.

In [ ]:
# Analizar el desbalance de clases de la variable 'inactivo'
counts = df['inactivo'].value_counts()
pct = df['inactivo'].value_counts(normalize=True) * 100

desbalance_df = pd.DataFrame({
    'Cantidad': counts,
    'Porcentaje (%)': pct
})
desbalance_df.index = desbalance_df.index.map({0: 'Activo (0)', 1: 'Inactivo (1)'})

print(desbalance_df.round(2))

In [ ]:
sns.set_theme(style='whitegrid')
fig, ax = plt.subplots(figsize=(16, 9))

sns.barplot(
    x=desbalance_df.index,
    y=desbalance_df['Porcentaje (%)'],
    hue=desbalance_df['Porcentaje (%)'].round(2),
    ax=ax
)

ax.set_title('Distribución de Listings Activos vs Inactivos', fontsize=14, pad=15)
ax.set_ylabel('Porcentaje (%)')
ax.set_xlabel('Estado del Listing')
ax.set_ylim(0, 100)

for p in ax.patches:
    ax.annotate(f'{p.get_height():.1f}%',
                (p.get_x() + p.get_width() / 2., p.get_height() + 1.5),
                ha='center', va='center', fontsize=11, color='black', weight='bold')

plt.tight_layout()
plt.show()

## Parte 2. Análisis univariado

`price` está en pesos argentinos. Con asimetría alta, la media es engañosa: reportar mediana
y trabajar con `log_price`.

In [ ]:
pd.DataFrame(df["price"].describe(percentiles=[.05, .25, .5, .75, .95, .99]).round(0)).T

In [ ]:
print(f"Asimetría price:     {df['price'].skew():.2f}")
print(f"Asimetría log_price: {df['log_price'].skew():.2f}")
print(f"Curtosis price:      {df['price'].kurt():.2f}")
print(f"Curtosis log_price:  {df['log_price'].kurt():.2f}")

el precio del dataset tiene una cola derecha extrema, y que la transformación logarítmica ayuda mucho pero no resuelve el problema por completo. Nota que .kurt() de pandas devuelve la curtosis en exceso (una normal vale 0), así que 954 es una cifra enorme.

In [ ]:
q1, q3 = df["price"].quantile([.25, .75])
iqr = q3 - q1
print(f"% outliers superiores (IQR): {(df['price'] > q3 + 1.5 * iqr).mean() * 100:.1f}%")
print(f"% con minimum_nights >= 28: {(df['minimum_nights'] >= 28).mean() * 100:.1f}%")
print(df["room_type"].value_counts(normalize=True).round(3))

## Parte 3. Análisis bivariado con tests de hipótesis

Como `price` no es normal se usan tests no paramétricos. Con muestras grandes casi todo test
da p muy pequeño: reportar siempre el **tamaño de efecto**.

In [ ]:
# 3.1 Precio por tipo de alojamiento: Kruskal-Wallis
grupos = [g["price"].values for _, g in df.groupby("room_type")]
H, p = stats.kruskal(*grupos)
print(f"Kruskal-Wallis price ~ room_type: H={H:.1f}, p={p:.2e}")
print(df.groupby("room_type")["price"].agg(["count", "median"]).round(0))

In [ ]:
# 3.2 Precio: inactivos vs activos (Mann-Whitney + rank-biserial)
a = df.loc[df["inactivo"] == 1, "price"]
b = df.loc[df["inactivo"] == 0, "price"]
U, p = stats.mannwhitneyu(a, b, alternative="two-sided")
r_rb = 2 * U / (len(a) * len(b)) - 1
print(f"Mann-Whitney: p={p:.2e}, rank-biserial={r_rb:.3f}")

# 3.3 Inactividad vs tipo de alojamiento: chi-cuadrado + V de Cramer
tabla = pd.crosstab(df["room_type"], df["inactivo"])
chi2, p, dof, _ = stats.chi2_contingency(tabla)
v = np.sqrt(chi2 / (tabla.values.sum() * (min(tabla.shape) - 1)))
print(f"Chi2 inactivo ~ room_type: p={p:.2e}, V de Cramer={v:.3f}")

In [ ]:
# 3.4 Correlación de Spearman
NUMS = ["price", "minimum_nights", "number_of_reviews",
        "calculated_host_listings_count", "availability_365", "number_of_reviews_ltm"]

corr_matrix = df[NUMS].corr(method="spearman").round(2)

plt.figure(figsize=(8, 6))
sns.heatmap(
    corr_matrix,
    annot=True,
    cmap="coolwarm",
    fmt=".2f",
    vmin=-1,
    vmax=1,
    linewidths=0.5
)
plt.title("Matriz de Correlación de Spearman", fontsize=14, pad=15)
plt.tight_layout()
plt.show()

In [ ]:
# 3.5 Barrios (n >= 30): precio mediano y tasa de inactividad
barrios = (df.groupby("neighbourhood")
             .agg(n=("id", "count"),
                  precio_mediano=("price", "median"),
                  tasa_inactivo=("inactivo", "mean"))
             .query("n >= 30")
             .sort_values("precio_mediano", ascending=False))
print(barrios.head(10).round(2))
print(barrios.tail(10).round(2))

# 3.6 Profesionalización: hosts con varios listings vs uno solo
df.groupby(df["calculated_host_listings_count"] > 1)["inactivo"].mean().round(3)

## Kruskal-Wallis (KW)

**Qué pregunta responde.** Dado un conjunto de grupos independientes (aquí, 4 tipos de alojamiento), ¿alguno tiende a tener valores distintos de los demás?

- **H0:** todos los grupos provienen de la misma distribución.
- **H1:** al menos un grupo tiende a tener valores mayores o menores.

**Cómo funciona, paso a paso:**

1. Se juntan todos los precios (N = 27.634) y se ordenan de menor a mayor, asignando rangos 1, 2, 3... Los empates reciben el rango promedio.
2. Se calcula el rango promedio de cada grupo.
3. Si H0 fuera cierta, el rango promedio de cada grupo sería parecido al global, que es (N+1)/2.
4. El estadístico H mide cuánto se alejan los rangos promedio de ese valor, ponderando por el tamaño de cada grupo:

```text
H = 12 / (N (N+1)) * suma_i ( R_i^2 / n_i ) - 3 (N+1)

R_i = suma de rangos del grupo i;  n_i = tamaño del grupo i
```

5. Bajo H0, H sigue aproximadamente una chi-cuadrado con k-1 grados de libertad (aquí 3). Con 3 gl, el valor crítico al 5% es 7,81, y el tuyo es 3.168,6.

**Por qué lo usamos en lugar de ANOVA:**
- `price` tiene asimetría 29,7 y curtosis 954, y ANOVA compara medias, que esos extremos distorsionan.
- Los grupos están muy desbalanceados (24.920 frente a 126), lo que vuelve frágil el supuesto de varianzas iguales.
- KW trabaja con rangos, así que un precio de 5 millones pesa lo mismo que uno apenas más alto que el anterior.

**Tus resultados.**
- `p = 0.00e+00` no es literalmente cero. Es un *underflow* numérico, y debe reportarse como p < 1e-300.
- Los medianas ordenan los grupos así: Hotel (253.616) > Entire home/apt (110.075) > Private room (52.836) > Shared room (30.577). El hotel cuesta unas 2,3 veces el departamento entero, y este unas 2,1 veces la habitación privada.
- El tamaño de efecto se calcula con epsilon cuadrado, y da aproximadamente 0,115:

```python
N = sum(len(g) for g in grupos)
eps2 = H / (N - 1)
print(f"epsilon^2 = {eps2:.3f}")   # ~0.115: efecto moderado (0.01 chico, 0.06 mediano, 0.14 grande)
```

El tipo de alojamiento explica alrededor del 11% de la variación de rangos del precio. Es un efecto real y moderado, no total: dentro de cada tipo sigue habiendo mucha dispersión (barrio, tamaño, ubicación).

**Qué NO dice KW.** Solo indica que hay diferencias en algún lado, no entre cuáles grupos. Para eso se necesita un test post hoc con corrección por comparaciones múltiples:

Con 126 y 151 observaciones, Hotel y Shared room tienen medianas menos precisas, así que conviene reportar su `count` junto al resultado.

## Mann-Whitney (inactivos frente a activos)

Es el equivalente de KW para **dos** grupos. También trabaja con rangos y compara si los valores de un grupo tienden a ser mayores que los del otro.

**Tus resultados:** p = 2,45e-17 y rank-biserial = -0,072.

- **El p-valor** dice que la diferencia es estadísticamente significativa. Con casi 28.000 filas, hasta diferencias pequeñas dan p muy bajos.
- **El tamaño de efecto** dice cuánto importa. Con r = 2U/(n1·n2) - 1 = -0,072, se despeja U/(n1·n2) = 0,464. Esa es la probabilidad de que un listing inactivo elegido al azar sea más caro que uno activo, y es menor que 0,5. Por lo tanto, los inactivos tienden a ser ligeramente más baratos.
- **La magnitud** es muy pequeña: |r| = 0,07 está por debajo del umbral convencional de efecto chico (0,1). Es la ilustración típica de "significativo pero casi irrelevante en la práctica".

**Conclusión:** el precio por sí solo discrimina muy poco entre activos e inactivos. El signo negativo depende de que `a` (inactivos) sea la primera muestra.

## Chi-cuadrado y V de Cramér (inactividad frente a tipo de alojamiento)

**Qué pregunta responde.** ¿Son independientes dos variables categóricas?

- **H0:** la proporción de inactivos es la misma en todos los tipos de alojamiento.
- **Cómo funciona:** se compara la tabla observada con la que se esperaría bajo independencia (esperado = total de fila × total de columna / N). El estadístico suma (observado - esperado)² / esperado en todas las celdas, y se compara con una chi-cuadrado con (filas-1)(columnas-1) = 3 gl.

**Resultados.** p = 1,19e-163 significa que la asociación no es casualidad. La V de Cramér es el tamaño de efecto, en una escala de 0 a 1: con 1 grado de libertad efectivo, 0,1 es chico, 0,3 mediano y 0,5 grande. Tu 0,165 queda entre chico y mediano, una asociación real pero modesta.

**Lo que falta:** el test no dice qué tipos de alojamiento se desvían de lo esperado. Estas celdas lo muestran, y además verifican el supuesto de frecuencias esperadas de al menos 5, que los grupos pequeños podrían violar:

## Lectura conjunta

| Test | Variables | Evidencia | Efecto | Lectura |
| --- | --- | --- | --- | --- |
| Kruskal-Wallis | price ~ room_type | p < 1e-300 | epsilon² ≈ 0,115 (moderado) | El tipo de alojamiento estructura el precio. |
| Mann-Whitney | price ~ inactivo | p = 2e-17 | r = -0,07 (muy chico) | El precio casi no separa activos de inactivos. |
| Chi-cuadrado | inactivo ~ room_type | p = 1e-163 | V = 0,165 (chico a mediano) | El tipo de alojamiento se asocia modestamente con la inactividad. |

***

**Criterio general de decisión.** Se usan dos filtros: el p-valor (menor a 0,05) para descartar el azar, y el tamaño de efecto para decidir si la diferencia merece atención. En este dataset todos los p son diminutos por el tamaño muestral, así que el que discrimina es el efecto.

**Implicancia para el modelo.** Como cada variable por separado se asocia débilmente con la inactividad, es razonable esperar que la capacidad predictiva del clasificador sea modesta y que dependa de combinaciones de variables. Eso es una hipótesis, y la Parte 5 la confirmará o la refutará.

**Una limitación común a los tres tests.** Suponen observaciones independientes, y aquí no lo son del todo: un mismo anfitrión puede tener hasta 20 listings (`calculated_host_listings_count`), con precios y comportamientos parecidos. Eso hace que los p-valores sean más optimistas de lo real, aunque los tamaños de efecto siguen siendo informativos.

Si me pasas el resultado del post hoc de Dunn y de los residuos, te ayudo a redactar la conclusión por tipo de alojamiento.

## Parte 4. Preparación y decisión anti-leakage

| Variable | Decisión | Motivo |
| --- | --- | --- |
| `number_of_reviews_ltm` | Excluir | Es el target mismo. |
| `reviews_per_month` | Excluir | Derivada de la actividad reciente; nula exactamente cuando no hay reseñas. |
| `last_review` | Excluir | Revela directamente si hubo reseñas recientes. |
| `id`, `host_id`, `host_profile_id`, `name`, `host_name` | Excluir | Identificadores o texto libre. |
| `number_of_reviews` | Incluir, con discusión | Histórica y no contiene el target, pero está correlacionada. |

**Confusor.** Un listing nuevo sin reseñas aparece como inactivo sin serlo realmente.
Evaluar el modelo con y sin `number_of_reviews` para medir la dependencia.

In [ ]:
# Para reducir cardinalidad -> (regla basada en frecuencia, no en el target)
conteo = df["neighbourhood"].map(df["neighbourhood"].value_counts())
df.loc[conteo < 30, "neighbourhood"] = "Otros"

In [ ]:
from sklearn.model_selection import train_test_split

NUM = ["price", "minimum_nights", "number_of_reviews",
       "calculated_host_listings_count", "availability_365", "latitude", "longitude"]
CAT = ["room_type", "neighbourhood"]

X, y = df[NUM + CAT], df["inactivo"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
print(f"Inactivos Train: {y_train.mean():.3f} | Test: {y_test.mean():.3f}")

## Parte 5. Baselines por validación cruzada (solo train)

Todo se compara con CV sobre train: evaluar baselines en test contaminaría la selección de modelo.
El Dummy da AUC cercano a 0,5 y PR-AUC igual a la proporción de inactivos. Si dos modelos
difieren en menos de un desvío estándar, la diferencia no es concluyente.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder

from sklearn.dummy import DummyClassifier
from sklearn.ensemble import (
    RandomForestClassifier,
    AdaBoostClassifier,
    GradientBoostingClassifier,
    HistGradientBoostingClassifier
    )
from sklearn.model_selection import StratifiedKFold, cross_validate

In [ ]:
SEED=42

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

def make_pipe(model):
    pre = ColumnTransformer(
        [("cat", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1), CAT)],
        remainder="passthrough",
    )
    return Pipeline([("pre", pre), ("model", model)])

# Tras el ColumnTransformer, las categóricas quedan primero y luego las numéricas
CAT_MASK = [True] * len(CAT) + [False] * len(NUM)

modelos = {
    "Dummy (prior)": DummyClassifier(strategy="prior"),
    "Random Forest (Bagging)": RandomForestClassifier(random_state=42, n_jobs=1),
    "AdaBoost": AdaBoostClassifier(random_state=42),
    "Gradient Boosting (Hist)": HistGradientBoostingClassifier(
        categorical_features=CAT_MASK,
        early_stopping=True, n_iter_no_change=20, random_state=42,
    ),
}

In [ ]:
filas = []
for nombre, m in modelos.items():
    r = cross_validate(make_pipe(m), X_train, y_train, cv=cv,
                       scoring=["roc_auc", "average_precision"])
    filas.append({
        "modelo": nombre,
        "auc_media": r["test_roc_auc"].mean(),
        "auc_std": r["test_roc_auc"].std(),
        "pr_auc_media": r["test_average_precision"].mean(),
    })


In [ ]:
resultados = pd.DataFrame(filas).sort_values("auc_media", ascending=False).round(4)
resultados

## Parte 6. Optimización con RandomizedSearchCV

In [ ]:
from sklearn.experimental import enable_halving_search_cv
from sklearn.model_selection import HalvingRandomSearchCV

DISTS = {
    "Random Forest (Bagging)": {
        "model__n_estimators": [200, 400],
        "model__max_depth": [None, 10, 20],
        "model__min_samples_leaf": [1, 3, 5],
        "model__max_features": ["sqrt", 0.5],
    },
    "AdaBoost": {
        "model__n_estimators": [100, 200, 400],
        "model__learning_rate": [0.05, 0.1, 0.5, 1.0],
    },
    "Gradient Boosting (Hist)": {
        "model__learning_rate": [0.03, 0.05, 0.1],
        "model__max_iter": [300, 600],          # early stopping corta antes si hace falta
        "model__max_leaf_nodes": [15, 31, 63],
        "model__min_samples_leaf": [20, 50, 100],
        "model__l2_regularization": [0.0, 1.0],
    },
}

ganador = resultados[resultados["modelo"] != "Dummy (prior)"].iloc[0]["modelo"]
print("Modelo a optimizar:", ganador)


busqueda = HalvingRandomSearchCV(
    make_pipe(modelos[ganador]),
    param_distributions=DISTS[ganador],
    n_candidates=24,
    factor=3,
    resource="n_samples",
    min_resources=2000,     # el valor por defecto es demasiado chico
    scoring="roc_auc",
    cv=cv,
    random_state=42,
    n_jobs=-1,
)
busqueda.fit(X_train, y_train)
best = busqueda.best_estimator_

print("Mejores hiperparámetros:", busqueda.best_params_)
print(f"AUC CV del mejor: {busqueda.best_score_:.4f}")

## Parte 7. Umbral de decisión con predicciones out-of-fold

El umbral por defecto (0,5) rara vez es adecuado con clases desbalanceadas.
Se elige con predicciones out-of-fold sobre train, nunca sobre test.

In [ ]:
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import precision_recall_curve

MIN_PREC = 0.60   # precisión mínima exigida (regla de negocio)

oof = cross_val_predict(best, X_train, y_train, cv=cv, method="predict_proba")[:, 1]
prec, rec, thr = precision_recall_curve(y_train, oof)

ok = prec[:-1] >= MIN_PREC
if ok.any():
    umbral = thr[ok].min()          # menor umbral que cumple la precisión => máximo recall
else:
    f1 = 2 * prec[:-1] * rec[:-1] / (prec[:-1] + rec[:-1] + 1e-12)
    umbral = thr[np.argmax(f1)]
    print("Ningún umbral alcanza la precisión mínima; se usa el de mayor F1.")

print(f"Umbral elegido: {umbral:.3f}")

## Parte 8. Evaluación final en test (una sola vez)

In [ ]:
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    classification_report,
    roc_curve,
    precision_recall_curve
)
import matplotlib.pyplot as plt
import seaborn as sns

# Obtener probabilidades y predicciones
y_prob = best.predict_proba(X_test)[:, 1]
y_pred = (y_prob >= umbral).astype(int)

print(f"AUC-ROC test: {roc_auc_score(y_test, y_prob):.4f}")
print(f"PR-AUC test:  {average_precision_score(y_test, y_prob):.4f}")
print(classification_report(y_test, y_pred, digits=3))

# Configurar subplots
sns.set_theme(style="whitegrid")
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# 1. Curva ROC
fpr, tpr, _ = roc_curve(y_test, y_prob)
auc_score = roc_auc_score(y_test, y_prob)
sns.lineplot(x=fpr, y=tpr, ax=ax1, color="darkorange", label=f"ROC curve (AUC = {auc_score:.4f})", linewidth=2.5)
ax1.plot([0, 1], [0, 1], color="navy", linestyle="--", linewidth=1.5)
ax1.set_xlim([0.0, 1.0])
ax1.set_ylim([0.0, 1.05])
ax1.set_xlabel("Tasa de Falsos Positivos (FPR)", fontsize=11)
ax1.set_ylabel("Tasa de Verdaderos Positivos (TPR)", fontsize=11)
ax1.set_title("Curva ROC", fontsize=13, pad=10)
ax1.legend(loc="lower right")

# 2. Curva Precision-Recall
prec, rec, _ = precision_recall_curve(y_test, y_prob)
pr_auc = average_precision_score(y_test, y_prob)
sns.lineplot(x=rec, y=prec, ax=ax2, color="blue", label=f"PR curve (AP = {pr_auc:.4f})", linewidth=2.5)
# Dibujar el umbral elegido en la curva
idx_umbral = np.argmin(np.abs(_ - umbral))
ax2.plot(rec[idx_umbral], prec[idx_umbral], 'ro', markersize=10, label=f'Umbral elegido ({umbral:.3f})')

ax2.set_xlim([0.0, 1.0])
ax2.set_ylim([0.0, 1.05])
ax2.set_xlabel("Exhaustividad (Recall)", fontsize=11)
ax2.set_ylabel("Precisión (Precision)", fontsize=11)
ax2.set_title("Curva Precision-Recall", fontsize=13, pad=10)
ax2.legend(loc="lower left")

plt.tight_layout()
plt.show()

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

# Graficar la matriz de confusión de manera visual y sencilla
disp = ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred,
    display_labels=['Activo (0)', 'Inactivo (1)'],
    cmap='Blues',
    values_format='d'
)

plt.grid(False) # Desactivar líneas de cuadrícula para ver mejor el gráfico
plt.title('Matriz de Confusión - Conjunto de Test')
plt.show()

In [ ]:
imp = permutation_importance(best, X_test, y_test, scoring="roc_auc",
                             n_repeats=5, random_state=42, n_jobs=-1)

(pd.Series(imp.importances_mean, index=X_test.columns).sort_values(ascending=False).round(4))

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Crear un DataFrame con las importancias para graficar
importancia_df = pd.DataFrame({
    'Variable': X_test.columns,
    'Importancia': imp.importances_mean
}).sort_values('Importancia', ascending=False)

# Configurar estilo y graficar
sns.set_theme(style='whitegrid')
plt.figure(figsize=(10, 6))

sns.barplot(
    x='Importancia',
    y='Variable',
    data=importancia_df,
    hue='Variable',
    palette='viridis',
    orient='h'
)

plt.title('Importancia de las Variables por Permutación (ROC-AUC)', fontsize=14, pad=15)
plt.xlabel('Disminución media en ROC-AUC')
plt.ylabel('Variable')
plt.tight_layout()
plt.show()

## Hipótesis a verificar

| Hipótesis | Dónde verificarla |
| --- | --- |
| `price` es muy asimétrico y `log_price` corrige gran parte de la asimetría. | Parte 2 |
| Tipo de alojamiento y barrio explican diferencias de precio con efecto grande. | Parte 3.1 y 3.5 |
| Los inactivos tienen menos `number_of_reviews` históricas y más `availability_365`. | Parte 3.4 y 8 |
| Hosts con un solo listing y hosts profesionales tienen distinta tasa de inactividad. | Parte 3.6 |
| Los ensambles superan claramente al Dummy en AUC y PR-AUC. | Parte 5 |
| Las diferencias entre los tres ensambles son menores a un desvío estándar. | Parte 5, `auc_std` |
| Quitar `number_of_reviews` baja el AUC de forma notable. | Repetir Partes 5 a 8 |

## Rúbrica

1. **Leakage:** tuning y umbral solo con train; sin `number_of_reviews_ltm`, `reviews_per_month` ni `last_review` como features.
2. **Métricas:** se justifica AUC-ROC y PR-AUC frente a accuracy según la proporción real de inactivos.
3. **Estadística:** tests no paramétricos con tamaño de efecto; se explica por qué no se usa t-test.
4. **Costos:** se argumenta qué error pesa más (no detectar un inactivo vs contactar a un host activo) y cómo eso justifica `MIN_PREC`.
5. **Confusor:** se discute el caso de listings nuevos sin reseñas y se prueba el modelo sin `number_of_reviews`.